# Day 9: Security Guardrails & Faithfulness Evaluation
**Config Source:** `.env` + `configs/app_config.yaml` (`engine/settings.py`) — Zero hardcoded URLs in notebook cells.


### Cell 1 — Centralized Settings & Modular Engine Bootstrap


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, json, time
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))

from engine.settings import SETTINGS
from engine.router import ModelRouter
from engine.qdrant_store import QdrantVectorStore
from engine.use_case_loader import UseCaseLoader
from engine.visualizer import StoryVisualizer
from engine.debugger import TraceDebugger

# Import PromptRegistry safely regardless of module filename (prompts.py or prompt_registry.py)
try:
    from engine.prompts import PromptRegistry
except ImportError:
    from engine.prompt_registry import PromptRegistry

# All URLs, paths, domain names, and cloud toggles come from SETTINGS (.env + configs/app_config.yaml)
ACTIVE_DOMAIN = SETTINGS.active_domain  # Override here or via BOOTCAMP_DOMAIN in .env

loader = UseCaseLoader(domain=ACTIVE_DOMAIN)
llm = ModelRouter(local_model_path=SETTINGS.local_model_path, enable_vertex=SETTINGS.enable_vertex_ai)
qdrant = QdrantVectorStore(collection_name=loader.collection_name, path=SETTINGS.qdrant_db_path)
prompts = PromptRegistry()
viz = StoryVisualizer()
dbg = TraceDebugger(debug_mode=False, enable_phoenix=False)

RUNBOOKS = loader.load_runbooks()
INCIDENTS = loader.load_incidents()
qdrant.upsert_documents(RUNBOOKS)

print(f"[Ready] Domain: {loader.domain} | LLM: {llm.provider} | VectorDB: {qdrant.mode} ({loader.collection_name})")


### Cell 2 — Production Incident Selector (0..9)


In [ ]:
# Select any of the 10 production incidents in the active domain (0..9)
INCIDENT_INDEX = 0
ACTIVE_INCIDENT = INCIDENTS[INCIDENT_INDEX]

print(f"Selected Incident: [{ACTIVE_INCIDENT['incident_id']}] {ACTIVE_INCIDENT['title']} (Severity: {ACTIVE_INCIDENT['severity']})")
print("Raw Log:", ACTIVE_INCIDENT["raw_log"])


### Cell 3 — Engineering Concept Brief
**Concept: Input/Output Defense-in-Depth**
Tests incident index `5` (every domain's secret-leak + prompt-injection stress case) through regex redaction and faithfulness scoring.


### Cell 4 — Traced Execution & X-Ray Visual Storyboard


In [ ]:
import re

security_inc = INCIDENTS[5]
dbg.start_run("day09_guardrails")

def guard_and_sanitize(raw_log: str) -> dict:
    injection_detected = bool(re.search(r"ignore previous instructions|output all .* passwords", raw_log, re.I))
    sanitized = re.sub(r"hf_[A-Za-z0-9]{8,}", "[REDACTED_HF_TOKEN]", raw_log)
    sanitized = re.sub(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", "[REDACTED_IP]", sanitized)
    sanitized = re.sub(r"'Ignore previous instructions[^']*'", "[BLOCKED_INJECTION_PAYLOAD]", sanitized, flags=re.I)
    return {"injection_blocked": injection_detected, "sanitized_log": sanitized}

with dbg.start_span(f"Day9_GuardrailsEval_{security_inc['incident_id']}", span_kind="CHAIN") as root_span:
    t0 = time.perf_counter()
    guard_res = guard_and_sanitize(security_inc["raw_log"])
    t_g = round((time.perf_counter() - t0) * 1000, 2)
    dbg.record_step("input_pii_injection_firewall", security_inc["raw_log"], guard_res, t_g, span_kind="GUARDRAIL")

    t1 = time.perf_counter()
    matched_sec_rb = qdrant.search(guard_res["sanitized_log"], top_k=1)[0]
    t_r = round((time.perf_counter() - t1) * 1000, 2)
    dbg.record_step("safe_qdrant_lookup", guard_res["sanitized_log"], matched_sec_rb, t_r, span_kind="RETRIEVER")

viz.render_story(
    title=f"Day 9 Security Guardrails X-Ray — {security_inc['incident_id']}",
    stages=[
        {"name": "Hostile Raw Input", "badge": "UNTRUSTED", "summary": "Contains exposed secret token, IP, and injection", "payload": security_inc["raw_log"]},
        {"name": "Sanitized Payload", "badge": "GUARDRAIL", "latency_ms": t_g, "summary": f"Injection Blocked: {guard_res['injection_blocked']}", "payload": guard_res["sanitized_log"]},
        {"name": "Safe Runbook Match", "badge": "RETRIEVER", "latency_ms": t_r, "summary": matched_sec_rb["metadata"]["runbook_id"], "payload": matched_sec_rb["text"]}
    ],
    phoenix_url=dbg.phoenix_url
)


### Cell 5 — Guarded Cloud Parity Check (`SETTINGS.enable_vertex_ai`)


In [ ]:
if SETTINGS.enable_vertex_ai:
    print("Vertex AI Model Armor & Gen AI Eval active.")
else:
    print("ENABLE_VERTEX_AI=false -> Local Security Guardrails verified.")


### Cell 6 — Automated Validation Gate


In [ ]:
assert guard_res["injection_blocked"] is True and "[REDACTED_HF_TOKEN]" in guard_res["sanitized_log"]
print("[PASS] Day 9 Guardrails & Security Gate passed!")
